# Model Distillation for Computer Vision
## Part 3: Training YOLO26 Target Model

**Workshop Duration:** 2 hours  
**Level:** Intermediate

---

### Learning Objectives
By the end of this notebook, you will:
1. Train a YOLO26 model on auto-labeled data
2. Evaluate model performance with standard metrics
3. Export the model for edge deployment

---

## 1. Setup & Dataset Verification

In [ ]:
import torch
import os
from pathlib import Path
import yaml

# Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Verify Ultralytics version supports YOLO26
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

# Need >= 8.4.0 for YOLO26
version_parts = ultralytics.__version__.split('.')
major, minor = int(version_parts[0]), int(version_parts[1])
assert (major > 8) or (major == 8 and minor >= 4), \
    "Please upgrade: uv pip install ultralytics>=8.4.0 for YOLO26 support"

from ultralytics import YOLO

In [ ]:
# Verify our dataset from Notebook 2
dataset_path = Path("../data/labeled_dataset")
yaml_path = dataset_path / "data.yaml"

if yaml_path.exists():
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
    
    print("Dataset configuration:")
    print(f"  Classes: {config.get('names', [])}")
    print(f"  Number of classes: {config.get('nc', 'unknown')}")
    
    # Count images
    train_images = list((dataset_path / "train" / "images").glob("*.jpg"))
    val_images = list((dataset_path / "valid" / "images").glob("*.jpg")) if (dataset_path / "valid").exists() else []
    
    print(f"  Training images: {len(train_images)}")
    print(f"  Validation images: {len(val_images)}")
else:
    print("ERROR: Dataset not found. Please run Notebook 2 first.")
    print(f"Expected path: {yaml_path}")

---

## 2. YOLO26 Model Selection

YOLO26 offers 5 variants with different accuracy/speed tradeoffs:

| Model | Parameters | mAP@50-95 | CPU (ms) | T4 TensorRT (ms) | Use Case |
|-------|------------|-----------|----------|------------------|----------|
| **yolo26n** | 2.4M | 40.9 | 38.9 | 1.7 | Edge/Mobile |
| **yolo26s** | 9.5M | 48.6 | 87.2 | 2.5 | Balanced |
| **yolo26m** | 20.4M | 53.1 | 220.0 | 4.7 | Production |
| **yolo26l** | 24.8M | 55.0 | 286.2 | 6.2 | High Accuracy |
| **yolo26x** | 55.7M | 57.5 | 525.8 | 11.8 | Maximum Accuracy |

For this workshop, we'll use **YOLO26n** (nano) for fast training, then optionally try **YOLO26s** for comparison.

In [ ]:
# Load YOLO26 nano model (pretrained on COCO)
model = YOLO("yolo26n.pt")

print(f"Model: {model.model_name}")
print(f"Task: {model.task}")

---

## 3. Training Configuration

### Key Training Parameters

In [ ]:
# Training configuration
TRAINING_CONFIG = {
    # Dataset
    "data": str(yaml_path),
    
    # Training duration
    "epochs": 50,           # Start with 50 for workshop (increase for production)
    "patience": 10,          # Early stopping patience
    
    # Batch & image size
    "batch": 16,             # Adjust based on GPU memory
    "imgsz": 640,            # Standard YOLO input size
    
    # Hardware
    "device": 0,             # GPU index (use -1 for CPU)
    "workers": 8,            # Data loading workers
    
    # Optimization
    "optimizer": "auto",     # YOLO26 uses MuSGD by default
    "lr0": 0.01,             # Initial learning rate
    
    # Output
    "project": "../runs",
    "name": "distilled_model",
    "save": True,
    "plots": True,
}

print("Training configuration:")
for k, v in TRAINING_CONFIG.items():
    print(f"  {k}: {v}")

### GPU Memory Guidelines

If you encounter CUDA out of memory errors:
1. Reduce `batch` size (try 8 or 4)
2. Reduce `imgsz` (try 416)
3. Use `batch="auto"` to auto-scale

---

## 4. Training

Let's train our distilled model!

In [ ]:
# Start training
print("Starting training...")
print("This may take 10-15 minutes depending on GPU.")
print("=" * 50)

results = model.train(**TRAINING_CONFIG)

print("\n" + "=" * 50)
print("Training complete!")

In [ ]:
# Display training results
print("Training Results:")
print("=" * 40)

if hasattr(results, 'results_dict'):
    for metric, value in results.results_dict.items():
        if isinstance(value, float):
            print(f"  {metric}: {value:.4f}")

### Understanding Training Metrics

| Metric | Description | Target |
|--------|-------------|--------|
| **box_loss** | Bounding box regression loss | Lower is better |
| **cls_loss** | Classification loss | Lower is better |
| **mAP@50** | Mean AP at 50% IoU threshold | > 0.5 good, > 0.7 great |
| **mAP@50-95** | Mean AP across IoU thresholds | More strict metric |

In [ ]:
# Display training curves
from IPython.display import Image as IPImage, display

results_dir = Path(TRAINING_CONFIG["project"]) / TRAINING_CONFIG["name"]
curves_path = results_dir / "results.png"

if curves_path.exists():
    display(IPImage(filename=str(curves_path), width=800))
else:
    print(f"Training curves not found at {curves_path}")

---

## 5. Model Evaluation

In [ ]:
# Load the best trained weights
best_weights = results_dir / "weights" / "best.pt"

if best_weights.exists():
    trained_model = YOLO(str(best_weights))
    print(f"Loaded best model from: {best_weights}")
else:
    print("Best weights not found, using last checkpoint")
    trained_model = model

In [ ]:
# Run validation
print("Running validation...")
metrics = trained_model.val(data=str(yaml_path))

print("\nValidation Results:")
print("=" * 40)
print(f"  mAP@50: {metrics.box.map50:.4f}")
print(f"  mAP@50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

In [ ]:
# Display confusion matrix
confusion_matrix_path = results_dir / "confusion_matrix.png"

if confusion_matrix_path.exists():
    display(IPImage(filename=str(confusion_matrix_path), width=600))
else:
    print("Confusion matrix not found")

---

## 6. Inference with Trained Model

In [ ]:
# Run inference on test images
import matplotlib.pyplot as plt
from PIL import Image

# Get a test image
test_images = list((dataset_path / "train" / "images").glob("*.jpg"))[:3]

if test_images:
    fig, axes = plt.subplots(1, len(test_images), figsize=(15, 5))
    if len(test_images) == 1:
        axes = [axes]
    
    for ax, img_path in zip(axes, test_images):
        # Run inference
        results = trained_model(str(img_path))
        
        # Plot annotated image
        annotated = results[0].plot()
        ax.imshow(annotated)
        ax.axis('off')
        ax.set_title(img_path.name)
    
    plt.tight_layout()
    plt.show()
else:
    print("No test images found")

In [ ]:
# Measure inference speed
import time
import numpy as np

if test_images:
    # Warm-up
    _ = trained_model(str(test_images[0]))
    
    # Benchmark
    times = []
    for _ in range(10):
        start = time.time()
        _ = trained_model(str(test_images[0]))
        times.append((time.time() - start) * 1000)  # Convert to ms
    
    print(f"Inference speed (YOLO26n):")
    print(f"  Mean: {np.mean(times):.2f} ms")
    print(f"  Std:  {np.std(times):.2f} ms")
    print(f"  FPS:  {1000 / np.mean(times):.1f}")

---

## 7. Model Export for Deployment

YOLO26 supports multiple export formats for different deployment targets.

In [ ]:
# Export to ONNX (universal format)
print("Exporting to ONNX...")
onnx_path = trained_model.export(format="onnx")
print(f"Exported to: {onnx_path}")

In [ ]:
# Check model sizes
import os

pytorch_size = os.path.getsize(best_weights) / (1024 * 1024)
onnx_size = os.path.getsize(onnx_path) / (1024 * 1024) if os.path.exists(onnx_path) else 0

print("Model sizes:")
print(f"  PyTorch (.pt): {pytorch_size:.2f} MB")
print(f"  ONNX (.onnx):  {onnx_size:.2f} MB")

### Export Format Options

| Format | Use Case | Command |
|--------|----------|--------|
| **ONNX** | Universal, cloud deployment | `export(format="onnx")` |
| **TensorRT** | NVIDIA GPUs (fastest) | `export(format="engine")` |
| **TFLite** | Mobile (Android/iOS) | `export(format="tflite")` |
| **OpenVINO** | Intel hardware | `export(format="openvino")` |
| **CoreML** | Apple devices | `export(format="coreml")` |

In [ ]:
# Optional: Export to TensorRT (if available)
# Uncomment to run
# print("Exporting to TensorRT...")
# tensorrt_path = trained_model.export(format="engine")
# print(f"Exported to: {tensorrt_path}")

---

## 8. Exercise: Compare Model Sizes

Try training with different YOLO26 variants and compare the results.

In [ ]:
# EXERCISE: Train with YOLO26s (small) and compare
# Uncomment and run to compare models

# model_s = YOLO("yolo26s.pt")
# 
# results_s = model_s.train(
#     data=str(yaml_path),
#     epochs=50,
#     batch=16,
#     imgsz=640,
#     device=0,
#     project="../runs",
#     name="distilled_model_s",
# )
# 
# print("Comparison:")
# print(f"  YOLO26n mAP@50: {metrics.box.map50:.4f}")
# print(f"  YOLO26s mAP@50: {results_s.results_dict.get('metrics/mAP50(B)', 'N/A')}")

---

## 9. The Distillation Value Proposition

Let's summarize what we achieved:

In [ ]:
# Summary
print("="*60)
print("MODEL DISTILLATION RESULTS")
print("="*60)
print("")
print("BASE MODEL (Teacher):")
print("  Model: Grounding DINO")
print("  Size: ~1.5 GB")
print("  Inference: ~80-120ms per image")
print("  Capability: Open-vocabulary detection")
print("")
print("TARGET MODEL (Student):")
print(f"  Model: YOLO26n")
print(f"  Size: {pytorch_size:.2f} MB")
print(f"  Inference: {np.mean(times):.2f}ms per image")
print(f"  mAP@50: {metrics.box.map50:.4f}")
print("")
print("DISTILLATION BENEFITS:")
print(f"  Size reduction: ~{1500/pytorch_size:.0f}x smaller")
print(f"  Speed improvement: ~{100/np.mean(times):.0f}x faster")
print("  Edge-deployable: Yes")
print("  Human annotation: 0 images")
print("="*60)

---

## 10. Summary & Next Steps

### What We Accomplished

1. Trained a **YOLO26n** model on auto-labeled data
2. Achieved **good accuracy** with **zero human annotation**
3. Created an **edge-deployable** model (~100x smaller than base)
4. Exported to **ONNX** for universal deployment

### Key Takeaways

| Aspect | Traditional | Distillation |
|--------|-------------|-------------|
| Annotation cost | $500-$10,000 | $50-$200 compute |
| Time to model | Weeks | 1-2 hours |
| Expected accuracy | 85-95% | 70-85% |
| Human effort | High | Minimal |

### Production Recommendations

1. **More data:** Use 500-1000+ images per class
2. **Longer training:** 100-200 epochs
3. **Label refinement:** Review and fix auto-labels
4. **Model selection:** Choose based on deployment constraints
5. **Continuous improvement:** Iterate with real-world feedback

### Resources

- [Ultralytics YOLO26 Docs](https://docs.ultralytics.com/models/yolo26/)
- [Autodistill Documentation](https://docs.autodistill.com/)
- [Roboflow Universe](https://universe.roboflow.com/) - Sample datasets

---

**Workshop complete! Thank you for participating.**